In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score, classification_report
from openai import OpenAI
from tqdm import tqdm
import os
import time
import json
# -------------------------------
# 1. Ler datasets
# -------------------------------

# JSON pode ser .jsonl (um json por linha). Ajuste conforme seu caso:
try:
    df_model = pd.read_json("gpt-4o-mini_results.json", lines=True)
except:
    df_model = pd.read_json("gpt-4o-mini_results.json")

#display(df_model.head())

df_manual = pd.read_csv("manual_evaluation.csv")

# -------------------------------
# 2. Cruzar os datasets pelo id
# -------------------------------

df_merged = df_manual.merge(
    df_model[["id", "classification"]],
    on="id",
    how="inner"
)

print("Tamanho do conjunto avaliado:", len(df_merged))
display(df_merged.head())
# Debug
#display(df_merged.head())

# -------------------------------
# 3. Preparar rótulos
# -------------------------------

# Normalizar (sim/nao → 1/0)
mapping = {
    "sim": 1,
    "não": 0, "nao": 0,
    "yes": 1, "no": 0  # caso tenha variações
}

df_merged["y_majority"] = df_merged["majority"].str.strip().str.lower().map(mapping)
df_merged["y_pred"] = df_merged["classification"].str.strip().str.lower().map(mapping)

# Remover linhas com erro de mapeamento
df_clean = df_merged.dropna(subset=["y_majority", "y_pred"])

y_majority = df_clean["y_majority"]
y_pred_gpt_4o_mini = df_clean["y_pred"]

# -------------------------------
# 4. Métricas
# -------------------------------

acc = accuracy_score(y_majority, y_pred_gpt_4o_mini)
f1 = f1_score(y_majority, y_pred_gpt_4o_mini)
conf = confusion_matrix(y_majority, y_pred_gpt_4o_mini)
kappa = cohen_kappa_score(y_majority, y_pred_gpt_4o_mini)

# -------------------------------
# 5. Exibir resultados
# -------------------------------

print("\n📌 RESULTADOS DA AVALIAÇÃO\n")
print(f"Acurácia: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

print("\nMatriz de Confusão (linhas = verdade, colunas = predição):")
print(conf)
#print(classification_report(y_true, y_pred, target_names=['não','sim']))
print(classification_report(y_majority, y_pred_gpt_4o_mini, target_names=['nao', 'sim']))


Tamanho do conjunto avaliado: 385


,id,title,description,subject_1,subject_2,subject_3,majority,classification
0,9JXwWeu7QwA,REAÇÃO DA VACINA 😭😭😭😭 #shortsvideo #vacination...,NaN,sim,sim,nao,sim,sim
1,8EdMfjqavtI,Vacina inédita contra a dengue: os bastidores ...,A dengue é considerada um grande desafio de sa...,sim,sim,sim,sim,sim
2,8F5f70qrupw,"CAPA CADERNETA VACINAS , protege e enfeita car...",NaN,nao,nao,nao,nao,nao
3,uDbVFPqP2JY,🌷 Caderneta de Vacina Personalizada | Delicada...,"Cada vacina, um cuidado… cada página, uma lemb...",nao,nao,nao,nao,nao
4,FqmbTkj4Adg,A vacinação em Porto Nacional foi um sucesso! ...,A vacinação em Porto Nacional foi um sucesso! ...,sim,sim,sim,sim,sim



📌 RESULTADOS DA AVALIAÇÃO

Acurácia: 0.8779
F1-score: 0.9153
Cohen's Kappa: 0.6970

Matriz de Confusão (linhas = verdade, colunas = predição):
[[ 84  18]
 [ 29 254]]
              precision    recall  f1-score   support

         nao       0.74      0.82      0.78       102
         sim       0.93      0.90      0.92       283

    accuracy                           0.88       385
   macro avg       0.84      0.86      0.85       385
weighted avg       0.88      0.88      0.88       385



In [2]:


PROMPT_BASE = """Você é um assistente especializado em classificação de conteúdo sobre saúde pública.

Tarefa: Leia o título e a descrição do vídeo e determine apenas se o assunto é relacionado a vacinação em seres humanos.

Importante: O objetivo não é avaliar se o conteúdo é verdadeiro ou falso, mas apenas se o vídeo trata semanticamente de vacinas reais aplicadas em pessoas. Mesmo que o vídeo traga teorias da conspiração, opiniões pessoais, críticas ou desinformação, se ele **fala sobre vacinas humanas**, a resposta deve ser **"Sim"**

Critérios de classificação:

Responda "Sim" se o vídeo:
- Fala sobre vacinas, vacinação ou imunização em seres humanos;
- Menciona campanhas de vacinação, políticas públicas ou temas como eficácia, segurança, obrigatoriedade ou efeitos adversos de vacinas;
- Aborda doenças preveníveis por vacinas (COVID-19, gripe, sarampo, HPV, etc.);
- Apresenta depoimentos, notícias, opiniões, ou debates sobre vacinas humanas — mesmo que contenha desinformação.

Responda "Não" se o vídeo:
- Usa a palavra “vacina” fora do contexto médico real (ex: “vacina para preguiça”);
- Fala de vacinas em animais;
- Usa “vacina” em contextos fictícios, humorísticos, metafóricos, ou de games;
- Cita “vacina” apenas de passagem, sem relação relevante com saúde pública ou imunização humana.

Avalie o vídeo:
"""

MODEL = "gpt-5-mini-2025-08-07"
OUTPUT_LOG = f'{MODEL}_log.txt'
OUTPUT_JSON = f"{MODEL}_sample_results.json"
BATCH_SIZE = 1
df = df_manual

done_ids = set()
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                done_ids.add(record['id'])
            except Exception:
                pass

client = OpenAI()

for i in tqdm(range(0, len(df))):
    row = df.iloc[i]

    if row['id'] in done_ids:
        continue

    videos_text = ""
    
    title = str(row['title']).strip()
    desc = str(row.get('description', '')).strip()
    videos_text += f"\n\n Vídeo {i+1}:\nTítulo: {title}\nDescrição: {desc}"

    full_prompt = PROMPT_BASE + videos_text
    #print(full_prompt[:1000])

    
    try:
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{'role':'user', 'content':full_prompt}]
        )
        text = response.choices[0].message.content.strip().lower()
        #print(text)
        with open(OUTPUT_LOG, 'a', encoding='utf-8') as f_log:
            f_log.write(f'\n\n-- Lote {i}: --\n')
            f_log.write(f"Título: {row['title']}\n")
            if row['description'] == row['description']:
                f_log.write(f"Descrição: {row['description']} \n")
            f_log.write(text)
            f_log.write('\n--------')

        
        reply = ''        
        for line in text.split('\n'):
            line = line.strip().lower()
            if 'sim' in line:
                reply = 'sim'
            elif 'não' in line or 'nao' in line:
                reply = 'nao'
        
        #print(replies)
        results = []
        video_id = row['id']
        if video_id in done_ids:
            continue
        
        res = {
            'id' : video_id,
            'classification' : reply,
            'title' : row['title'],
            'description' : row['description']
        }
        results.append(res)
        done_ids.add(video_id)
        
        with open(OUTPUT_JSON, 'a', encoding='utf-8') as f:
            for r in results:
                json.dump(r, f, ensure_ascii=False)
                f.write('\n')
        
        time.sleep(0.2)
    except Exception as e:
        print(f'erro {e}')
        time.sleep(2)


100%|█████████████████████████████████████████████████████████████████████████████| 385/385 [00:00<00:00, 48272.36it/s]


In [8]:
kappa = cohen_kappa_score(y_pred_gpt_4_1, y_pred_gpt_4o_mini)
print(f' GPT-4o-mini vs GPT-4.1: Cohen\'s Kappa: {kappa:.3}')

 GPT-4o-mini vs GPT-4.1: Cohen's Kappa: 0.677


In [9]:
df_gpt_5_mini = pd.read_json('gpt-5-mini-2025-08-07_sample_results.json', lines=True)
y_pred_gpt_5_mini = df_gpt_5_mini['classification'].map({'sim' : 1, 'nao' : 0})

kappa = cohen_kappa_score(y_pred_gpt_5_mini, y_pred_gpt_4o_mini)
print(f' GPT-4o-mini vs GPT-5-mini: Cohen\'s Kappa: {kappa:.3}')

 GPT-4o-mini vs GPT-5-mini: Cohen's Kappa: 0.609


In [10]:
kappa = cohen_kappa_score(y_pred_gpt_4o_mini, y_majority)
print(f'GPT-4o-mini vs Maioria: Cohen\'s Kappa: {kappa:.3}')

GPT-4o-mini vs Maioria: Cohen's Kappa: 0.697


In [11]:
df_gpt = pd.read_json('gpt-4.1_sample_results.json', lines=True)
y_pred_gpt_4_1 = df_gpt['classification'].map({'sim' : 1, 'nao' : 0})

kappa = cohen_kappa_score(y_pred_gpt_4_1, y_pred_gpt_5_mini)
print(f'GPT-4.1 vs GPT-5-mini: Cohen\'s Kappa: {kappa:.3}')

GPT-4.1 vs GPT-5-mini: Cohen's Kappa: 0.89


In [12]:
df_gpt = pd.read_json('gpt-4.1_sample_results.json', lines=True)
y_pred_gpt_4_1 = df_gpt['classification'].map({'sim' : 1, 'nao' : 0})

kappa = cohen_kappa_score(y_pred_gpt_4_1, y_majority)
print(f'GPT-4.1 vs Maioria: Cohen\'s Kappa: {kappa:.3}')

GPT-4.1 vs Maioria: Cohen's Kappa: 0.697


In [13]:
df_gpt_5_mini = pd.read_json('gpt-5-mini-2025-08-07_sample_results.json', lines=True)
y_pred_gpt_5_mini = df_gpt_5_mini['classification'].map({'sim' : 1, 'nao' : 0})

kappa = cohen_kappa_score(y_pred_gpt_5_mini, y_majority)
print(f'GPT-5-mini vs Maioria: Cohen\'s Kappa: {kappa:.3}')

GPT-5-mini vs Maioria: Cohen's Kappa: 0.64
